# 3 · Interpretable scorecard and untouched test evaluation

Unweighted logistic regression estimates the cohort's probability of TARGET=1 from train-selected WoE features. Fixed L2 regularization controls correlated features. Native logistic probabilities are retained; no calibration or model tuning uses test outcomes. The exported score represents score 600 at good:bad odds 50:1, with 20 points doubling those odds.

In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown

# The executor supplies these paths. Interactive runs default to local exports.
ROOT = Path(os.environ.get('HOME_CREDIT_PROJECT_ROOT', str(Path.cwd()))).resolve()
if not (ROOT / 'src' / 'analysis.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis import ensure_analysis, load_features, eda_tables

INPUT = Path(os.environ.get('HOME_CREDIT_FEATURES', str(ROOT / 'exports' / 'applicant_features.csv')))
OUTPUT = Path(os.environ.get('HOME_CREDIT_ANALYSIS', str(ROOT / 'outputs' / 'analysis')))
STATUS = os.environ.get('HOME_CREDIT_DATA_STATUS', 'real')
LGD = float(os.environ.get('HOME_CREDIT_LGD', '0.45'))
SEED = int(os.environ.get('HOME_CREDIT_SEED', '42'))
manifest = ensure_analysis(INPUT, OUTPUT, data_status=STATUS, lgd=LGD, seed=SEED)
get_ipython().run_line_magic('matplotlib', 'inline')
display(Markdown(f'**Data status: {manifest["data_status"].upper()}**. '
    + ('Synthetic pipeline validation; no Home Credit conclusions.' if STATUS != 'real'
       else 'Computed from the supplied accepted-cohort export.')))


In [ ]:
metrics = pd.read_csv(OUTPUT / 'model_metrics.csv')
odds = pd.read_csv(OUTPUT / 'odds_ratios.csv')
bands = pd.read_csv(OUTPUT / 'risk_bands.csv')
calibration = pd.read_csv(OUTPUT / 'calibration.csv')
display(metrics)
display(odds.sort_values('coefficient'))
display(Markdown(f'Intercept: **{manifest["intercept"]:.4f}**. Selected features: **{len(manifest["selected_features"])}**.'))

## Interpret odds ratios correctly

Each odds ratio is exp(coefficient) for a **+1 WoE** change, conditional on the other included features. It is not a one-year age change or a one-currency-unit income change. The model uses regularized estimates, so the table does not present naive Wald confidence intervals. Coefficients can change sign with correlated predictors; they are not causal effects.

In [ ]:
display(Image(filename=str(OUTPUT / 'figures' / 'test_roc.png')))
display(Image(filename=str(OUTPUT / 'figures' / 'test_calibration.png')))
display(calibration[calibration.split == 'test'])

## Risk bands

Nine validation-prediction quantiles define ten increasing predicted-risk bands. Held-out observed rates are shown with Wilson intervals and may reverse. Test outcomes never trigger bin reordering, merging, recalibration or a refit. Tied predictions can produce empty bands. Higher risk_score means safer; higher risk_band means riskier.

In [ ]:
display(bands[bands.split.isin(['validation', 'test'])])
display(Image(filename=str(OUTPUT / 'figures' / 'risk_bands.png')))
violations = bands[bands.split.isin(['validation', 'test']) & bands.observed_rate_monotonicity_violation]
display(Markdown(f'Observed nonempty-band monotonicity reversals: **{len(violations)}**.'))
print('Validation-defined PD boundaries:', manifest['band_boundaries'])

## Validation limits

The 60/20/20 split is stratified and at applicant level, with no duplicate applicants. It is random because no usable calendar date of the current application is provided; it does not establish temporal performance. AUC, KS, Gini and Brier scores are point estimates for this cohort. Inspect sample sizes and calibration before considering any deployment.